In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Dataset path
CSV_PATH = r"C:\Users\LENOVO\Downloads\Crashbox_DL_Digital_Twin\data\raw\crashbox_spatial_temporal_data_FIXED.csv"

In [17]:
df = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


Dataset loaded successfully.
Shape: (1050, 7)


In [18]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
df.head()


Column names:
['Time', 'NodeLabel', 'X', 'Y', 'Z', 'U3', 'RF3_GLOBAL']

Data types:
Time          float64
NodeLabel       int64
X             float64
Y             float64
Z             float64
U3            float64
RF3_GLOBAL    float64
dtype: object

First 5 rows:


,Time,NodeLabel,X,Y,Z,U3,RF3_GLOBAL
0,0.0,1,-13.75,-10.0,150.0,0.0,-224.2386
1,0.0,2,-13.75,15.0,150.0,0.0,-224.2386
2,0.0,3,-13.75,15.0,0.0,0.0,-224.2386
3,0.0,4,-13.75,-10.0,0.0,0.0,-224.2386
4,0.0,5,-38.75,15.0,150.0,0.0,-224.2386


In [19]:
df.describe()


,Time,NodeLabel,X,Y,Z,U3,RF3_GLOBAL
count,1050.000000,1050.000000,1050.000000,1050.000000,1050.000000,1050.000000,1050.000000
mean,0.035008,25.500000,-15.750000,8.500000,69.000000,-37.587905,-15056.422433
std,0.021204,14.437746,6.785562,10.017263,56.098103,22.805331,3762.530331
min,0.000000,1.000000,-38.750000,-10.000000,0.000000,-76.635360,-19745.967000
25%,0.017502,13.000000,-13.750000,0.000000,15.000000,-56.492000,-16998.984000
50%,0.035005,25.500000,-13.750000,15.000000,62.500000,-37.515133,-15660.571000
75%,0.052513,38.000000,-13.750000,15.000000,125.000000,-18.741100,-14818.995000
max,0.070000,50.000000,-13.750000,15.000000,150.000000,0.000000,-224.238600


In [20]:
print("Number of unique nodes:", df["NodeLabel"].nunique())
print("Number of time steps:", df["Time"].nunique())

print("\nSamples per node (first 5 nodes):")
df.groupby("NodeLabel").size().head()


Number of unique nodes: 50
Number of time steps: 21

Samples per node (first 5 nodes):


NodeLabel
1    21
2    21
3    21
4    21
5    21
dtype: int64

In [21]:
print("Missing values per column:")
print(df.isnull().sum())

# Remove duplicates
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]

print(f"\nRemoved {before - after} duplicate rows")

# Remove invalid numeric values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

print("Final dataset shape:", df.shape)


Missing values per column:
Time          0
NodeLabel     0
X             0
Y             0
Z             0
U3            0
RF3_GLOBAL    0
dtype: int64

Removed 0 duplicate rows
Final dataset shape: (1050, 7)


In [22]:
INPUT_FEATURES = ["Time", "X", "Y", "Z", "U3"]
TARGET_FEATURE = "RF3_GLOBAL"


X = df[INPUT_FEATURES]
y = df[TARGET_FEATURE]

print("Input features:")
X.head()


Input features:


,Time,X,Y,Z,U3
0,0.0,-13.75,-10.0,150.0,0.0
1,0.0,-13.75,15.0,150.0,0.0
2,0.0,-13.75,15.0,0.0,0.0
3,0.0,-13.75,-10.0,0.0,0.0
4,0.0,-38.75,15.0,150.0,0.0


In [23]:
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))

print("Feature scaling completed.")


Feature scaling completed.


In [24]:
joblib.dump(scaler_X, r"C:\Users\LENOVO\Downloads\Crashbox_DL_Digital_Twin\data\processed\scaler_X.pkl")
joblib.dump(scaler_y, r"C:\Users\LENOVO\Downloads\Crashbox_DL_Digital_Twin\data\processed\scaler_y.pkl")

print("Scalers saved.")


Scalers saved.


In [25]:
np.random.seed(42)

nodes = df["NodeLabel"].unique()
np.random.shuffle(nodes)

split_ratio = 0.7
split_idx = int(len(nodes) * split_ratio)

train_nodes = nodes[:split_idx]
test_nodes = nodes[split_idx:]

train_df = df[df["NodeLabel"].isin(train_nodes)]
test_df = df[df["NodeLabel"].isin(test_nodes)]

print("Train nodes:", len(train_nodes))
print("Test nodes:", len(test_nodes))
print("Train samples:", train_df.shape[0])
print("Test samples:", test_df.shape[0])


Train nodes: 35
Test nodes: 15
Train samples: 735
Test samples: 315


In [26]:
X_train = train_df[INPUT_FEATURES]
y_train = train_df[TARGET_FEATURE]

X_test = test_df[INPUT_FEATURES]
y_test = test_df[TARGET_FEATURE]


In [27]:
X_train_scaled = scaler_X.transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

print("Train/Test scaling completed.")


Train/Test scaling completed.


In [28]:
BASE_PATH = r"C:\Users\LENOVO\Downloads\Crashbox_DL_Digital_Twin\data\processed"

pd.DataFrame(X_train_scaled, columns=INPUT_FEATURES).to_csv(f"{BASE_PATH}/X_train.csv", index=False)
pd.DataFrame(y_train_scaled, columns=["RF3"]).to_csv(f"{BASE_PATH}/y_train.csv", index=False)

pd.DataFrame(X_test_scaled, columns=INPUT_FEATURES).to_csv(f"{BASE_PATH}/X_test.csv", index=False)
pd.DataFrame(y_test_scaled, columns=["RF3"]).to_csv(f"{BASE_PATH}/y_test.csv", index=False)

print("✅ Preprocessed train/test datasets saved.")


✅ Preprocessed train/test datasets saved.


In [29]:
print("X_train shape:", X_train_scaled.shape)
print("y_train shape:", y_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)
print("y_test shape:", y_test_scaled.shape)



X_train shape: (735, 5)
y_train shape: (735, 1)
X_test shape: (315, 5)
y_test shape: (315, 1)
